In [3]:
%pip install opencv-python numpy shapely matplotlib scikit-image

  Using cached scipy-1.15.3-cp310-cp310-macosx_14_0_x86_64.whl.metadata (61 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 12.2 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 11.0 MB/s  0:00:00
Using cached scipy-1.15.3-cp310-cp310-macosx_14_0_x86_64.whl (25.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [scikit-image] [scikit-image]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import cv2
import numpy as np
camera = cv2.VideoCapture(1)
window_name = "Aperçu de la caméra"

try:
    if not camera.isOpened():
        raise SystemExit("Impossible d'ouvrir la caméra.")

    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    while True:
        ok, frame = camera.read()
        if not ok:
            raise SystemExit("Impossible de lire une image de la caméra.")

        cv2.imshow(window_name, frame)
        key = cv2.waitKey(1) & 0xFF
        if key in (ord("q"), 27):
            break
        if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
            break
finally:
    camera.release()
    cv2.destroyAllWindows()

2026-11-24 18:09:58.865 python[40796:3015493] WARNING: AVCaptureDeviceTypeExternal is deprecated for Continuity Cameras. Please use AVCaptureDeviceTypeContinuityCamera and add NSCameraUseContinuityCameraDeviceType to your Info.plist.


In [1]:
import cv2
import numpy as np
from shapely.geometry import Polygon

from vision import (initialize_camera, initialize_vision, detect_thymio,
                    GridNotDetectedError, camera_id, camera_width, fps,
                    warmup_frames, grid_side_mm)
from global_navigation import plan_path

ARENA_W = ARENA_H = grid_side_mm   # 700 mm, from vision (origin top left corner)


def draw_path(frame, waypoints, H):
    """Dessine le chemin (mm) sur l'image camera (pixels)."""
    if not waypoints:
        return frame
    to_camera = np.linalg.inv(H)                             # mm -> pixels
    pts_mm = np.float32([waypoints])
    pts_px = cv2.perspectiveTransform(pts_mm, to_camera)[0]
    pts_px = np.rint(pts_px).astype(int)
    cv2.polylines(frame, [pts_px], False, (0, 0, 255), 3)
    for p in pts_px:
        cv2.circle(frame, tuple(p), 6, (0, 0, 255), -1)
    return frame


def compute_path(obstacles, thymio, goal_mm):
    """Lance la global navigation depuis la pose du Thymio."""
    if goal_mm is None:
        print("Aucun coup a jouer")
        return None
    result = plan_path(obstacles, thymio[:2], goal_mm, ARENA_W, ARENA_H)   # (x, y) seulement
    if result["error"]:
        print(result["error"])
        return None
    print(f"Chemin : {result['length']:.0f} mm, waypoints : {result['waypoints']}")
    return result["waypoints"]


def main():
    cap = initialize_camera(camera_id, camera_width, fps, warmup_frames)
    dictionary = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
    detector = cv2.aruco.ArucoDetector(dictionary)

    try:
        # 1. Vision : grille, pieces, coup optimal (une fois)
        ret, frame = cap.read()
        H, obstacles_mm, goal_mm = initialize_vision(frame)
        obstacles = [Polygon(v) for v in obstacles_mm]

        waypoints = None
        while True:
            ret, frame = cap.read()
            if not ret:
                raise RuntimeError("Impossible de lire l'image de la camera")

            # 2. Position du Thymio (a chaque image)
            thymio = detect_thymio(frame, H, detector)

            # 3. Global navigation : une fois, des que le Thymio est vu
            if waypoints is None and thymio is not None:
                waypoints = compute_path(obstacles, thymio, goal_mm)

            # 4. Affichage du chemin sur l'image camera
            cv2.imshow("Global navigation", draw_path(frame.copy(), waypoints, H))

            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"):                 # quitter
                break
            if key == ord("p") and thymio is not None:   # replanifier (ex. apres un kidnapping)
                waypoints = compute_path(obstacles, thymio, goal_mm)
            if key == ord("r"):                 # reinitialiser la Vision (pieces deplacees)
                try:
                    H, obstacles_mm, goal_mm = initialize_vision(frame)
                    obstacles = [Polygon(v) for v in obstacles_mm]
                    waypoints = None
                except GridNotDetectedError as error:
                    print(error)

    except GridNotDetectedError as error:
        print(error)
        cv2.waitKey(0)
    finally:
        cap.release()
        cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

2026-11-24 18:12:09.965 python[40887:3019832] WARNING: AVCaptureDeviceTypeExternal is deprecated for Continuity Cameras. Please use AVCaptureDeviceTypeContinuityCamera and add NSCameraUseContinuityCameraDeviceType to your Info.plist.


Caméra initialisée
Résolution : 1920 x 1080
FPS configurés : 30.0
Chemin : 101 mm, waypoints : [(59.59065628051758, 35.804954528808594), (75.0, 75.0), (116.66666666666667, 116.66666666666667)]


KeyboardInterrupt: 